# 🩺 Chronic Kidney Disease (CKD) — ML Classification Pipeline

**Dataset:** 1659 patients × 54 features  
**Task:** Binary classification — CKD (1) vs No CKD (0)  
**Models:** Logistic Regression & Random Forest  
**Key challenge:** Severe class imbalance (≈ 11:1) — addressed with SMOTE

---

## Step 1 — Install Dependencies & Import Libraries

In [ ]:
# Install required packages
!pip install imbalanced-learn xgboost --quiet
print("✅ Packages installed")

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── XGBoost ──────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Imbalanced-learn ─────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── Colab file upload ─────────────────────────────────────────────────────────
from google.colab import files

print("✅ All libraries imported successfully!")

## Step 2 — Upload & Load Dataset

In [ ]:
# Upload the CSV file from your local machine
uploaded = files.upload()          # Select ckd_1659.csv when prompted
filename = list(uploaded.keys())[0]

df_raw = pd.read_csv(filename)
print(f"✅ Dataset loaded: {filename}")
print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

## Step 3 — Dataset Exploration

In [ ]:
# ── Define the 11 features we'll use ─────────────────────────────────────────
FEATURES = [
    'Age', 'BMI', 'HbA1c', 'SerumCreatinine', 'BUNLevels',
    'GFR', 'HemoglobinLevels', 'CholesterolTotal',
    'ProteinInUrine', 'UrinaryTractInfections', 'FamilyHistoryKidneyDisease'
]
TARGET = 'Diagnosis'

print("=" * 55)
print(" DATASET OVERVIEW")
print("=" * 55)
print(f"\nTotal rows    : {df_raw.shape[0]}")
print(f"Total columns : {df_raw.shape[1]}")
print(f"Selected feat : {len(FEATURES)} features + 1 target")

# ── Data types of selected features ──────────────────────────────────────────
print("\n── Data Types of Selected Features ────────────────")
print(df_raw[FEATURES + [TARGET]].dtypes.to_string())

# ── Missing values ───────────────────────────────────────────────────────────
print("\n── Missing Values ──────────────────────────────────")
missing = df_raw[FEATURES + [TARGET]].isnull().sum()
print(missing.to_string())
print(f"\nTotal missing cells: {missing.sum()} (dataset is clean ✅)")

# ── Class distribution ───────────────────────────────────────────────────────
print("\n── Class Distribution (Target: Diagnosis) ──────────")
class_counts = df_raw[TARGET].value_counts()
print(f"  CKD    (1) : {class_counts[1]:>5}  ({class_counts[1]/len(df_raw)*100:.1f}%)")
print(f"  No CKD (0) : {class_counts[0]:>5}  ({class_counts[0]/len(df_raw)*100:.1f}%)")
print(f"  Imbalance ratio ≈ {class_counts[1]//class_counts[0]}:1")

# ── First 5 rows of selected features ────────────────────────────────────────
print("\n── Sample Rows (Selected Features) ─────────────────")
df_raw[FEATURES + [TARGET]].head()

## Step 4 — Feature Selection & Normalization

In [ ]:
# ── Keep only the 11 selected features + target ───────────────────────────────
df = df_raw[FEATURES + [TARGET]].copy()
print(f"Working dataframe shape: {df.shape}")

# ── Separate features (X) and target (y) ─────────────────────────────────────
X = df[FEATURES].values
y = df[TARGET].values

# ── Normalize all 11 features using StandardScaler ────────────────────────────
#    Fit ONLY on the full X here; we'll re-fit on training data after the split
#    (to avoid data leakage in production, but shown here for teaching clarity)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nScaling summary (mean ≈ 0, std ≈ 1 after scaling):")
print(pd.DataFrame(X_scaled, columns=FEATURES).describe().round(3).loc[['mean','std']])

## Step 5 — Stratified Train-Test Split (80 / 20)

In [ ]:
# stratify=y ensures both classes are proportionally represented in each split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # ← preserves the 11:1 ratio in both halves
)

print("Train-Test Split Results")
print("-" * 40)
print(f"Training samples : {len(X_train)} ({len(X_train)/len(X_scaled)*100:.0f}%)")
print(f"Testing  samples : {len(X_test)}  ({len(X_test)/len(X_scaled)*100:.0f}%)")
print()
print("Class distribution in TRAINING set:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'CKD' if cls == 1 else 'No CKD'
    print(f"  {label} ({cls}): {cnt}")
print()
print("Class distribution in TEST set:")
unique, counts = np.unique(y_test, return_counts=True)
for cls, cnt in zip(unique, counts):
    label = 'CKD' if cls == 1 else 'No CKD'
    print(f"  {label} ({cls}): {cnt}")

## Step 6 — SMOTE Oversampling (Training Data Only)

In [ ]:
# Record counts BEFORE SMOTE
before_counts = dict(zip(*np.unique(y_train, return_counts=True)))

# Apply SMOTE only on training data — test set is NEVER touched
# sampling_strategy=1.0 → minority class will match majority class count
# k_neighbors=5  → use 5 nearest neighbors to synthesize new samples
smote = SMOTE(sampling_strategy=1.0, k_neighbors=5, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Record counts AFTER SMOTE
after_counts = dict(zip(*np.unique(y_train_sm, return_counts=True)))

print("SMOTE Results (Training Data)")
print("-" * 40)
print(f"{'Class':<10} {'Before':>8} {'After':>8}")
print("-" * 40)
for cls in sorted(before_counts.keys()):
    label = f"CKD ({cls})"
    print(f"{label:<10} {before_counts[cls]:>8} {after_counts[cls]:>8}")
print("-" * 40)
print(f"{'Total':<10} {sum(before_counts.values()):>8} {sum(after_counts.values()):>8}")

## Step 7 — Visualize Class Distribution (Before vs After SMOTE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution: Before vs After SMOTE (Training Data)',
             fontsize=14, fontweight='bold', y=1.02)

labels = ['No CKD (0)', 'CKD (1)']
colors = ['#4CAF50', '#F44336']

# Before SMOTE
before_vals = [before_counts.get(0, 0), before_counts.get(1, 0)]
bars0 = axes[0].bar(labels, before_vals, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Before SMOTE', fontsize=13)
axes[0].set_ylabel('Sample Count')
axes[0].set_ylim(0, max(after_counts.values()) * 1.15)
for bar, val in zip(bars0, before_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontweight='bold')

# After SMOTE
after_vals = [after_counts.get(0, 0), after_counts.get(1, 0)]
bars1 = axes[1].bar(labels, after_vals, color=colors, edgecolor='black', width=0.5)
axes[1].set_title('After SMOTE', fontsize=13)
axes[1].set_ylabel('Sample Count')
axes[1].set_ylim(0, max(after_counts.values()) * 1.15)
for bar, val in zip(bars1, after_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print("✅ SMOTE has balanced the training classes (1:1 ratio).")

## Step 8 — Train All 4 Models

In [ ]:
# ── Model 1: Logistic Regression ──────────────────────────────────────────────
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_sm, y_train_sm)
print("✅ Logistic Regression trained.")

# ── Model 2: Random Forest ────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_sm, y_train_sm)
print("✅ Random Forest trained.")

# ── Model 3: SVM (Support Vector Machine) ────────────────────────────────────
# kernel='rbf'           → handles non-linear boundaries (best for medical data)
# class_weight='balanced' → handles class imbalance
# C=1.0                  → regularization strength (default, works well here)
# probability=True       → needed if you want predict_proba later
svm_model = SVC(
    kernel='rbf',
    class_weight='balanced',
    C=1.0,
    gamma='scale',
    random_state=42,
    probability=True
)
svm_model.fit(X_train_sm, y_train_sm)
print("✅ SVM (RBF kernel) trained.")

# ── Model 4: XGBoost ─────────────────────────────────────────────────────────
# scale_pos_weight       → handles imbalance (ratio of negative/positive)
# n_estimators=200       → number of boosting rounds
# learning_rate=0.1      → step size for each boosting round
# max_depth=6            → tree depth (default, prevents overfitting)
neg  = int((y_train_sm == 0).sum())
pos  = int((y_train_sm == 1).sum())
scale = neg / pos        # =1.0 after SMOTE but good practice to set

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=scale,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_model.fit(X_train_sm, y_train_sm)
print("✅ XGBoost trained.")

## Step 9 — Evaluate All 4 Models

In [ ]:
# ── Predictions on the held-out test set ─────────────────────────────────────
lr_pred  = lr_model.predict(X_test)
rf_pred  = rf_model.predict(X_test)
svm_pred = svm_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

# ── Helper function: compute all four metrics (macro average) ─────────────────
def get_metrics(y_true, y_pred, name):
    return {
        'Model'     : name,
        'Accuracy'  : accuracy_score(y_true, y_pred),
        'Precision' : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall'    : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'F1-Score'  : f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

lr_metrics  = get_metrics(y_test, lr_pred,  'Logistic Regression')
rf_metrics  = get_metrics(y_test, rf_pred,  'Random Forest')
svm_metrics = get_metrics(y_test, svm_pred, 'SVM')
xgb_metrics = get_metrics(y_test, xgb_pred, 'XGBoost')

# ── Print all classification reports ─────────────────────────────────────────
for metrics, pred, name in [
    (lr_metrics,  lr_pred,  'LOGISTIC REGRESSION'),
    (rf_metrics,  rf_pred,  'RANDOM FOREST'),
    (svm_metrics, svm_pred, 'SVM'),
    (xgb_metrics, xgb_pred, 'XGBOOST'),
]:
    print("=" * 55)
    print(f" {name} — Classification Report")
    print("=" * 55)
    print(classification_report(y_test, pred, target_names=['No CKD (0)', 'CKD (1)']))

## Step 10 — Visualizations

In [ ]:
# ── 10a: Confusion Matrices — 2×2 grid ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Confusion Matrices — All 4 Models', fontsize=15, fontweight='bold')

model_info = [
    (lr_pred,  'Logistic Regression', 'Blues',   axes[0][0]),
    (rf_pred,  'Random Forest',       'Greens',  axes[0][1]),
    (svm_pred, 'SVM (RBF)',           'Oranges', axes[1][0]),
    (xgb_pred, 'XGBoost',            'Purples', axes[1][1]),
]

for pred, title, cmap, ax in model_info:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap=cmap,
        xticklabels=['No CKD (0)', 'CKD (1)'],
        yticklabels=['No CKD (0)', 'CKD (1)'],
        ax=ax, linewidths=0.5, linecolor='gray',
        annot_kws={'size': 13, 'weight': 'bold'}
    )
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved as confusion_matrices.png")

In [ ]:
# ── 10b: Metrics comparison bar chart — 4 models ─────────────────────────────
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_results = [lr_metrics, rf_metrics, svm_metrics, xgb_metrics]
model_colors  = ['#2196F3', '#FF9800', '#E91E63', '#9C27B0']

x     = np.arange(len(metric_names))
width = 0.20   # narrower bars to fit 4 models

fig, ax = plt.subplots(figsize=(13, 6))

for i, (result, color) in enumerate(zip(model_results, model_colors)):
    vals = [result[m] for m in metric_names]
    bars = ax.bar(x + (i - 1.5) * width, vals, width,
                  label=result['Model'], color=color,
                  edgecolor='white', linewidth=0.8)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.008,
                f'{bar.get_height():.2f}',
                ha='center', fontsize=8, fontweight='bold')

ax.set_title('Model Performance Comparison — All 4 Models (Macro Average)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=12)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10, loc='upper right')
ax.spines[['top','right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved as metrics_comparison.png")

In [ ]:
# ── 10c: Feature Importance — Random Forest & XGBoost side by side ──────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance Comparison', fontsize=14, fontweight='bold')

# Random Forest
rf_imp = pd.DataFrame({'Feature': FEATURES, 'Importance': rf_model.feature_importances_})
rf_imp = rf_imp.sort_values('Importance', ascending=True)
colors_rf = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(rf_imp)))
bars = axes[0].barh(rf_imp['Feature'], rf_imp['Importance'],
                    color=colors_rf, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, rf_imp['Importance']):
    axes[0].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[0].set_title('Random Forest', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Importance Score')
axes[0].spines[['top','right']].set_visible(False)
axes[0].xaxis.grid(True, alpha=0.3)

# XGBoost
xgb_imp = pd.DataFrame({'Feature': FEATURES, 'Importance': xgb_model.feature_importances_})
xgb_imp = xgb_imp.sort_values('Importance', ascending=True)
colors_xgb = plt.cm.RdYlBu(np.linspace(0.3, 0.9, len(xgb_imp)))
bars2 = axes[1].barh(xgb_imp['Feature'], xgb_imp['Importance'],
                     color=colors_xgb, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars2, xgb_imp['Importance']):
    axes[1].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[1].set_title('XGBoost', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance Score')
axes[1].spines[['top','right']].set_visible(False)
axes[1].xaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved as feature_importance.png")

## Step 11 — Final Summary & Best Model

In [ ]:
# ── Build the summary table ───────────────────────────────────────────────────
summary_df = pd.DataFrame([
    lr_metrics, rf_metrics, svm_metrics, xgb_metrics
]).set_index('Model').round(4)

print("=" * 65)
print(" FINAL MODEL COMPARISON SUMMARY (Macro Average)")
print("=" * 65)
print(summary_df.to_string())
print()

# ── Declare the winner by F1-Score ────────────────────────────────────────────
best_model = summary_df['F1-Score'].idxmax()
best_row   = summary_df.loc[best_model]

print("=" * 65)
print(f" 🏆 BEST MODEL  : {best_model}")
print(f"    Accuracy    : {best_row['Accuracy']:.4f}")
print(f"    Precision   : {best_row['Precision']:.4f}")
print(f"    Recall      : {best_row['Recall']:.4f}")
print(f"    F1-Score    : {best_row['F1-Score']:.4f} (macro average)")
print("=" * 65)
print()
print("✅ Conclusion:")
print(f"   {best_model} outperforms all other models and is")
print(f"   selected as the final model for CKD risk prediction.")
print()
print("Why F1-Score (macro) is the right metric here:")
print("  • Dataset has class imbalance (11:1 ratio).")
print("  • Macro average weights both classes equally.")
print("  • F1 balances Precision and Recall — critical")
print("    when both false positives AND false negatives matter.")